# Paper R6: refits explained on each split's 500-row core (H3)

Conference paper, Phase 5 (D33, D35, D37 and its amendments, D38, D39; claude/paper/phase4_design.md sections 1, 3, 9 item 11, 12 items 5, 6, 11 and 13). **Runs on the PC only.**

What it does, in this order:
1. **Gates (Cell 3):** data, Stage 4 freeze, frozen pipelines, Stage 5 predictions, Stage 1 index files, every R4 file against `SHA256SUMS_r4.txt`, every R5 file against `SHA256SUMS_r5.txt`, the R5 models against their reproduction records, the split-1 Stage 3 `cv_results` files.
2. **Checks against stored outputs, before any refit (Cell 6; design section 12 item 11):** every R5 model reproduces its stored core outputs; the R6 code on the first 5 core rows of each R5 model reproduces the stored R5 SHAP values (permutation exactly, tree/linear within 1e-10); the worker path for permutation rows gives byte-identical values (if not, that model runs single-process).
3. **Seeds (Cell 7):** a new append-only 16th stream `paper_r6`; every refit seed, every bootstrap sample hash, the Rashomon set and the RF 200-row subset are derived and written to `seeds_r6.json` **before any fit**. On a resume the file must be identical.
4. **Refits and SHAP (Cell 9), in this order:** (a) bootstrap refits of CAT, XGB, LGBM, MLP — split 1: 10, splits 2–5: 5 (within-class resampling of the development rows with replacement; tuned hyperparameters and estimator seed fixed); (b) split 1: bootstrap refits of LR and RF (10 each); (c) split 1: seed-only refits (5 each of XGB, LGBM, CAT, MLP, RF; full development data, new estimator seed); (d) split 1: Rashomon set (Stage 3 configurations within 0.005 mean CV AP of the best, at most 10 per model including the selected one, which is not refitted; full development data). Bootstrap refits are explained in the **common** setting (decisive for H3) and the **native** setting; seed-only and Rashomon refits in the native setting only. Same cores, backgrounds and per-row permutation seeds as R5. MLP native = common: computed once, written to both files.
5. **RF fallback (section 9 item 11):** if the native TreeSHAP of the **first RF refit** takes more than 8 min with 6 workers, every later RF task (refits, seed-only, Rashomon; native and common) uses the fixed stratified 200-row subset of the core (28 positives); the decision is saved in `rf_fallback.json`.
6. **Outputs** in `paper_r6_outputs/` (never in `paper_r5_outputs/`): per-row SHAP files (17 digits) with `.done.json`, refit records, additivity, timing, environment log. Agreement statistics (H3) are computed later in the workspace.

**Resumable:** each refit is saved with its hash; each SHAP task is saved in row chunks under `_work/`. Run All again after a stop: finished refits and chunks are skipped and the outputs are byte-identical to an uninterrupted run (tested). Do not change the settings in Cell 2 between sessions.

**Run:** put the notebook in `thesis_xai/` (next to `paper_r4_outputs/`, `paper_r5_outputs/` with its `models/` folder, `r4_upload/SHA256SUMS_r4.txt` and `r5_upload/SHA256SUMS_r5.txt`), set `R6_CONFIRMED = True` in Cell 2, keep the laptop on mains power with sleep disabled, and Run All. Then run `python pack_r6_outputs.py` and upload `r6_upload/`.

**Time on the PC (planning only, from timing_r5.csv):** about 5.7 h with the permutation workers (about 6.9 h single-process), up to about 8.5 h with ×1.5 throttling (r6_time_estimate.py; Rashomon set at its upper bound of 9 refits per model); the decisive part (a) is about 2.4 h of it and runs first. If the RF fallback is triggered, about 1.3 h less. Cell 8 prints the plan; Cell 9 prints the time remaining. Disk: about 3 GB for the refitted models (mostly RF).

In [1]:
# Cell 1 - environment guard (S10): before numpy / scikit-learn are imported
import os, sys
TEST_MODE = os.environ.get("R6_TEST_MODE") == "1"          # Claude's workspace test only; never set on the PC
PROJECT_DIR = os.environ.get("R6_PROJECT_DIR", os.path.abspath("."))
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
import env_setup
env_setup.ensure_pinned(PROJECT_DIR, install=False)

All pinned packages present at the pinned versions.


{}

In [2]:
# Cell 2 - configuration
R6_CONFIRMED = True                       # set True to run (D39)
if TEST_MODE:
    R6_CONFIRMED = True
if not R6_CONFIRMED:
    raise SystemExit("Set R6_CONFIRMED = True in Cell 2, then Run All.")

import contextlib, hashlib, io, json, math, platform, time, warnings
from datetime import datetime, timezone
import joblib
import numpy as np
import pandas as pd
import shap
import sklearn
from joblib import Parallel, delayed
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
import thesis_common as tc

ENV = lambda k, d: os.environ.get(k, d) if TEST_MODE else d
FREEZE_SHA256 = "1dd5bf06708d148399675d9fc657857df1c86f015cbd83c24e79e8754b48c0d8"   # S4-047
PRED_SHA256 = "b9813461cd03e4552b50c245c8750648d08b9c48c1960d249642982bc15114ed"     # S5-136
SUMS_R4_SHA256 = ENV("R6_TEST_SUMS_R4", "dc79a5bb2476664230310c68ad9befcf6a6c8a3160891b75b28b81cae1c55b12")
SUMS_R5_SHA256 = ENV("R6_TEST_SUMS_R5", "e73e9812e8191cb07671d8363beeb6b7763b267a8075b61d6ba80cad7de2e9dc")   # D38
MANIFEST_R4_SHA256 = "d04472a61f148527087be432731f13e4344e4e7e40a3026ed3c52e3764b45ca1"
STAGE1_SHA = {"dev_folds.csv": "c71320238e98f1d8fedf1ce72475f8572e27524805201480857374ccc3ab7977",
              "shap_background_500_dev.csv": "981fe21fcd3e157857be057b0a9e43a93115fe99e10aff4313cbddd26e60a21c",
              "shap_perm_background_100_dev.csv": "a5c697d56d71fa26074110fef9e18f16cdfd3fbdff6c11741249c1bd6f466e09",
              "shap_perm_core_500_test.csv": "59dae554426fe5adbbc9330505ccfaa099b9022d9e8e7678d97a877952599fe9"}
FEATURES = list(tc.FEATURES)
MEMBERS = ["CAT", "XGB", "LGBM", "MLP"]                    # H2/H3 members (O33)
SPLIT1_EXTRA = ["LR", "RF"]                                # split 1 only, exploratory (B1 b)
SEED_MODELS = ["XGB", "LGBM", "CAT", "MLP", "RF"]          # LR is unchanged by the seed
RASH_MODELS = ["LR", "XGB", "LGBM", "MLP", "RF", "CAT"]
TREE = ["XGB", "LGBM", "RF", "CAT"]
SCALE_NATIVE = {"LR": "log-odds", "XGB": "log-odds", "LGBM": "log-odds", "CAT": "log-odds", "RF": "probability", "MLP": "probability"}
SPLITS = [int(s) for s in ENV("R6_SPLITS", "1,2,3,4,5").split(",")]
N_BOOT = {k: (10 if k == 1 else 5) for k in [1, 2, 3, 4, 5]}
N_SEED, RASH_MAX, RASH_MARGIN = 5, 10, 0.005               # Rashomon: at most 10 configurations per model, incl. the selected one
if TEST_MODE:                                              # test only: fewer refits
    N_BOOT = {k: min(v, int(ENV("R6_N_BOOT", "99"))) for k, v in N_BOOT.items()}
    N_SEED, RASH_MAX = min(N_SEED, int(ENV("R6_N_SEED", "99"))), min(RASH_MAX, int(ENV("R6_RASH_MAX", "99")))
RF_FALLBACK_S = float(ENV("R6_RF_FALLBACK_S", "480"))     # section 9 item 11: 8 min with 6 workers
RF_SUBSET_N, RF_SUBSET_POS = 200, 28
CAT_THREADS = 12                                           # D35 item 1
MAX_EVALS = 250
TREE_CHUNK, PERM_CHUNK = 50, 25
TREE_WORKERS = int(ENV("R6_TREE_WORKERS", str(max(1, min(6, os.cpu_count() or 2))))) if TEST_MODE else 6
# Permutation rows in worker processes (allowed by the per-row seeds) only for models whose prediction is single-threaded
# in R5 (cpu/wall about 1-1.5): CAT, MLP, RF, LR. XGB and LGBM already use about 11 cores in one process. Cell 6 checks
# on the PC that the worker path gives byte-identical values; a model that fails runs single-process.
PERM_WORKERS = {"CAT": 4, "MLP": 6, "RF": 6, "LR": 6}
if TEST_MODE:
    w = int(ENV("R6_PERM_WORKERS", "2"))
    PERM_WORKERS = {m: min(v, w) for m, v in PERM_WORKERS.items()} if w > 0 else {}
REG_ROWS = 5 if not TEST_MODE else int(ENV("R6_REG_ROWS", "3"))
MAX_ROWS = int(ENV("R6_MAX_ROWS", "0"))                    # 0 = all core rows (test only)
STOP_AFTER = ENV("R6_STOP_AFTER", "")                      # test only: "task:chunk" or "fit:unit" interrupt

# Seeds. S2 (13 streams) + 14th (shap_permutation) + 15th (paper_outer) + 16th (paper_r6, new, D39). Append-only:
# the first 15 words are asserted unchanged. Per split k: SeedSequence([paper_r6, k]).generate_state(17), words
#   0-9  bootstrap resampling seeds of refits 1-10 (splits 2-5 use 0-4)
#   10-14 estimator seeds of the split-1 seed-only refits 1-5 (mod 2**31 - 1, as MODEL_SEED)
#   15   RF fallback 200-row subset of the split-1 core
#   16   H2 core-row bootstrap (post-hoc, exploratory; r5_h2_boot.py; not used here)
SS16 = np.random.SeedSequence(tc.MASTER_SEED).generate_state(16)
SS15 = np.random.SeedSequence(tc.MASTER_SEED).generate_state(15)
assert [int(v) for v in SS16[:15]] == [int(v) for v in SS15], "seed derivation changed"
assert [int(v) for v in SS15[:13]] == [tc.SEEDS[k] for k in tc.SEED_STREAMS], "seed derivation changed"
assert int(SS15[13]) == 3957024168, "14th stream (shap_permutation) changed"
PAPER_OUTER, PAPER_R6 = int(SS15[14]), int(SS16[15])
assert PAPER_R6 == 1741519417, "16th stream (paper_r6) changed"
SPLIT_STREAMS = ["partition", "folds", "subsample_50k", "shap_background", "shap_perm_background", "test_core", "eval_bootstrap"]
STAGE6_PERM_SEED = int(np.random.SeedSequence(int(SS15[13])).generate_state(2)[0])
assert STAGE6_PERM_SEED == 2941821818, "Stage 6 permutation seed changed"


def r6_words(k):
    w = np.random.SeedSequence([PAPER_R6, k]).generate_state(17)
    assert [int(v) for v in w[:16]] == [int(v) for v in np.random.SeedSequence([PAPER_R6, k]).generate_state(16)]
    return [int(v) for v in w]


def perm_seed(k):                                          # as R5 (D37 amendment B5)
    if k == 1:
        return STAGE6_PERM_SEED
    s8 = np.random.SeedSequence([PAPER_OUTER, k]).generate_state(8)
    assert [int(v) for v in s8[:7]] == [int(v) for v in np.random.SeedSequence([PAPER_OUTER, k]).generate_state(7)]
    return int(s8[7])


def row_seed(base, row_id):                                # S15k: one seed per explained row
    return int(np.random.SeedSequence([int(base), int(row_id)]).generate_state(1)[0])


OUT = os.path.join(PROJECT_DIR, ENV("R6_OUT", "paper_r6_TEST") if TEST_MODE else "paper_r6_outputs")
R4 = os.path.join(PROJECT_DIR, "paper_r4_outputs")
R5 = ENV("R6_R5_DIR", os.path.join(PROJECT_DIR, "paper_r5_outputs"))
if TEST_MODE and not os.path.isabs(R5):
    R5 = os.path.join(PROJECT_DIR, R5)
WORK, MODELS_DIR = os.path.join(OUT, "_work"), os.path.join(OUT, "models")
for d in [OUT, WORK, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)
MACHINE = "workspace-test" if TEST_MODE else "pc"
if not TEST_MODE and (platform.system() != "Windows" or os.cpu_count() != 12):
    raise SystemExit("R6 runs on the project PC only (Windows, 12 logical CPUs): XGBoost and CatBoost refits differ between machines.")
print(f"machine={MACHINE} python={platform.python_version()} shap={shap.__version__} tree workers={TREE_WORKERS} "
      f"permutation workers={PERM_WORKERS} splits={SPLITS} refits={N_BOOT} seed-only={N_SEED} Rashomon max={RASH_MAX}")
print("paper_r6 =", PAPER_R6, "; permutation seeds:", {k: perm_seed(k) for k in SPLITS})

C:\Users\USER\Downloads\thesis_xai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


machine=pc python=3.13.0 shap=0.52.0 tree workers=6 permutation workers={'CAT': 4, 'MLP': 6, 'RF': 6, 'LR': 6} splits=[1, 2, 3, 4, 5] refits={1: 10, 2: 5, 3: 5, 4: 5, 5: 5} seed-only=5 Rashomon max=10
paper_r6 = 1741519417 ; permutation seeds: {1: 2941821818, 2: 1798040441, 3: 1261013684, 4: 3055778493, 5: 2471599976}


In [3]:
# Cell 3 - gates: data, freeze, Stage 5, Stage 1 index files, R4 files, R5 files and models, Stage 3 cv_results
def sha(path):
    return tc.sha256_file(path)


def read_json(p):
    with open(p, encoding="utf-8") as f:
        return json.load(f)


GATE = []
def gate(name, ok, detail=""):
    GATE.append({"check": name, "pass": bool(ok), "detail": str(detail)})
    if not ok:
        print("FAIL", name, detail)


X_all, y_all = tc.load_data(PROJECT_DIR)                                     # SHA-256 checked
FZ_PATH = os.path.join(PROJECT_DIR, "stage4_outputs", "frozen_models.json")
gate("Stage 4 freeze", sha(FZ_PATH) == FREEZE_SHA256)
FZ = json.load(open(FZ_PATH, encoding="utf-8"))
ALL6 = ["LR", "XGB", "LGBM", "MLP", "RF", "CAT"]
PIPE1 = {m: os.path.join(PROJECT_DIR, FZ["models"][m]["pipeline"]) for m in ALL6}
for m in ALL6:
    gate(f"split 1 frozen pipeline {m}", os.path.exists(PIPE1[m]) and sha(PIPE1[m]) == FZ["models"][m]["pipeline_sha256"])
PRED_PATH = os.path.join(PROJECT_DIR, "stage5_outputs", "test_predictions.csv")
gate("Stage 5 test_predictions.csv", os.path.exists(PRED_PATH) and sha(PRED_PATH) == PRED_SHA256)
PRED = pd.read_csv(PRED_PATH, float_precision="round_trip")
IDX1 = os.path.join(PROJECT_DIR, "stage1_outputs", "indices")
for f, h in STAGE1_SHA.items():
    gate(f"Stage 1 {f}", os.path.exists(os.path.join(IDX1, f)) and sha(os.path.join(IDX1, f)) == h)


def check_sums(label, paths, want_sha, root, skip_notebook):
    p = next((q for q in paths if os.path.exists(q)), None)
    gate(f"SHA256SUMS_{label}.txt found and unchanged", p is not None and sha(p) == want_sha, p)
    if p is None:
        return
    n = ok = 0
    for line in open(p, encoding="utf-8").read().splitlines():
        h, name = line.split("  ", 1)
        if ":" not in name:
            continue                                                       # the zips themselves
        rel = name.split(":", 1)[1]
        if rel == skip_notebook:
            continue                                                       # executed notebook: informative only
        n += 1
        good = os.path.exists(os.path.join(root, rel)) and sha(os.path.join(root, rel)) == h
        ok += good
        if not good:
            print("  differs or missing:", rel)
    gate(f"{label.upper()} files vs SHA256SUMS_{label}.txt", ok == n and n > 0, f"{ok}/{n}")


check_sums("r4", [os.path.join(PROJECT_DIR, "r4_upload", "SHA256SUMS_r4.txt"), os.path.join(R4, "SHA256SUMS_r4.txt"),
                  os.path.join(PROJECT_DIR, "SHA256SUMS_r4.txt")], SUMS_R4_SHA256, R4, "paper_r4_splits.ipynb")
MAN_PATH = os.path.join(R4, "indices", "MANIFEST_splits2_3_4_5.csv")
gate("MANIFEST_splits2_3_4_5.csv", os.path.exists(MAN_PATH) and sha(MAN_PATH) == MANIFEST_R4_SHA256)
MAN4 = pd.read_csv(MAN_PATH)
n_idx = sum(sha(os.path.join(R4, "indices", f"split{r.split}", r.file)) == r.sha256 for r in MAN4.itertuples())
gate("split 2-5 index files vs MANIFEST", n_idx == len(MAN4), f"{n_idx}/{len(MAN4)}")
check_sums("r5", [os.path.join(PROJECT_DIR, "r5_upload", "SHA256SUMS_r5.txt"), os.path.join(R5, "SHA256SUMS_r5.txt"),
                  os.path.join(PROJECT_DIR, "SHA256SUMS_r5.txt")], SUMS_R5_SHA256, R5, "paper_r5_shap.ipynb")

# R5 models (kept on the PC for R6): file hash = the R5 reproduction record (splits 2-5); split 1 = frozen pipelines
R5_MODEL = {}
for k in SPLITS:
    recs = {r["model"]: r for r in read_json(os.path.join(R5, f"split{k}", "reproduction", "done.json"))["models"]}
    for m in (MEMBERS + SPLIT1_EXTRA if k == 1 else MEMBERS):
        if k == 1:
            R5_MODEL[(k, m)] = PIPE1[m]
            gate(f"split 1 {m}: R5 used the frozen pipeline", recs[m]["model_sha256"] == FZ["models"][m]["pipeline_sha256"])
        else:
            p = os.path.join(R5, "models", f"split{k}_{m}.joblib")
            R5_MODEL[(k, m)] = p
            gate(f"split {k} {m}: R5 model file = reproduction record", os.path.exists(p) and sha(p) == recs[m]["model_sha256"]
                 and bool(recs[m]["exact"] or TEST_MODE), recs[m]["source"])

# split-1 Stage 3 cv_results (Rashomon set): 30 rows, mean = mean of the 5 fold APs, rank 1 = the frozen configuration
CV1 = {}
if 1 in SPLITS:
    for m in RASH_MODELS:
        p = os.path.join(PROJECT_DIR, "stage3_outputs", "models", m, f"cv_results_{m}.csv")
        ok = os.path.exists(p)
        if ok:
            cv = pd.read_csv(p, float_precision="round_trip")
            if "config_id" not in cv.columns:
                cv.insert(0, "config_id", np.arange(len(cv)))
            folds = cv[[f"split{j}_test_ap" for j in range(5)]].to_numpy()
            best = int(cv.loc[cv.rank_test_ap == 1, "config_id"].min())
            ok = (len(cv) == len(tc.sampled_configs(m)) and np.allclose(folds.mean(axis=1), cv.mean_test_ap, atol=1e-9)
                  and np.array_equal(cv.config_id.to_numpy(), np.arange(len(cv))) and best == FZ["models"][m]["config_id"])
            CV1[m] = cv
        gate(f"Stage 3 cv_results_{m}.csv (30 configurations, rank 1 = frozen config)", ok, sha(p) if os.path.exists(p) else "missing")
GATE_DF = pd.DataFrame(GATE)
print(GATE_DF.to_string(index=False))
if not GATE_DF["pass"].all():
    raise SystemExit("Input gates failed; nothing was fitted or explained. Report the FAIL lines.")

                                                                  check  pass                                                                detail
                                                         Stage 4 freeze  True                                                                      
                                             split 1 frozen pipeline LR  True                                                                      
                                            split 1 frozen pipeline XGB  True                                                                      
                                           split 1 frozen pipeline LGBM  True                                                                      
                                            split 1 frozen pipeline MLP  True                                                                      
                                             split 1 frozen pipeline RF  True                                   

In [4]:
# Cell 4 - helpers. model_output, tree_chunk, linear_all, Dedup and perm_rows are the R5 (= Stage 6) functions,
# unchanged; perm_chunk runs perm_rows in a worker process (one explainer per chunk; per-row seeds).
def write_atomic(path, blob):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        f.write(blob)
    os.replace(tmp, path)


def sha_bytes(b):
    return hashlib.sha256(b).hexdigest()


def model_output(m, pipe, X):
    clf = pipe.named_steps["clf"] if hasattr(pipe, "named_steps") else None
    if m in ("DT", "RF"):
        return clf.predict_proba(X)[:, 1]
    if m == "XGB":
        return clf.predict(X, output_margin=True).astype(float)
    if m == "LGBM":
        return clf.predict(X, raw_score=True)
    if m == "CAT":
        return clf.predict(X, prediction_type="RawFormulaVal")
    if m == "LR":
        return pipe.decision_function(X)
    return pipe.predict_proba(X)[:, 1]


def tree_chunk(path, m, Xc, Xbg, k):
    import time, warnings
    import joblib, numpy as np, shap
    t0, c0 = time.perf_counter(), time.process_time()
    pipe = joblib.load(path)
    clf = pipe.named_steps["clf"]
    if m in ("RF", "XGB", "LGBM"):
        clf.set_params(n_jobs=1)                            # CatBoost cannot change parameters after fitting
    obj = clf.get_booster() if m == "XGB" else clf          # shap 0.52 + xgboost 3.4.1: the sklearn wrapper fails (S15b)
    import contextlib, io
    with warnings.catch_warnings(), contextlib.redirect_stderr(io.StringIO()):
        warnings.simplefilter("ignore")
        ex = shap.TreeExplainer(obj, shap.maskers.Independent(Xbg, max_samples=len(Xbg)),
                                feature_perturbation="interventional", model_output="raw")
        e = ex(Xc, check_additivity=False)
        fx = model_output(m, pipe, Xc)
    v, b = np.asarray(e.values), np.asarray(e.base_values, dtype=float)
    if v.ndim == 3:
        v, b = v[:, :, 1], b[:, 1]
    b = np.broadcast_to(b, (len(Xc),)).astype(float)
    return k, v.astype(float), b, np.asarray(fx, float), time.process_time() - c0, time.perf_counter() - t0


def linear_all(path, Xc, Xbg):
    t0 = time.perf_counter()
    pipe = joblib.load(path)
    prep, clf = pipe.named_steps["prep"], pipe.named_steps["clf"]
    names = list(prep.get_feature_names_out())
    Zbg, Z = prep.transform(Xbg), prep.transform(Xc)
    e = shap.LinearExplainer(clf, shap.maskers.Independent(Zbg, max_samples=len(Zbg)))(Z)
    order = [names.index(f) for f in FEATURES]
    v = np.asarray(e.values)[:, order]
    b = np.broadcast_to(np.asarray(e.base_values, float), (len(Xc),)).astype(float)
    return v, b, model_output("LR", pipe, Xc), time.perf_counter() - t0


class Dedup:
    # exact cache of repeated rows within one call (the model is deterministic, so values are unchanged)
    def __init__(self, pipe):
        self.pipe = pipe

    def __call__(self, Z):
        U, inv = np.unique(Z, axis=0, return_inverse=True)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=DeprecationWarning)
            p = self.pipe.predict_proba(pd.DataFrame(U, columns=FEATURES))[:, 1]
        return p[inv.ravel()]


def perm_rows(ex, f, Xc, ids, seed):
    vals, base = [], []
    for x, r_ in zip(Xc, ids):
        np.random.seed(row_seed(seed, r_))                  # shap draws permutations from numpy's global generator
        e = ex(x[None, :], max_evals=MAX_EVALS, silent=True)
        vals.append(e.values[0]); base.append(float(np.asarray(e.base_values).ravel()[0]))
    return np.array(vals, float), np.array(base), f(Xc)


def load_pipe(path, m):
    pipe = joblib.load(path)
    if m == "RF":
        pipe.set_params(clf__n_jobs=1)                      # S12e: single-threaded RF prediction
    return pipe


def perm_explainer(path, m, Xbg_np):
    pipe = load_pipe(path, m); f = Dedup(pipe)
    return f, shap.PermutationExplainer(f, shap.maskers.Independent(Xbg_np, max_samples=len(Xbg_np)))


def perm_chunk(path, m, Xbg_np, Xc_np, ids, seed, k):
    # worker process: same functions, one explainer per chunk; values do not depend on chunking (per-row seeds)
    import time
    t0, c0 = time.perf_counter(), time.process_time()
    f, ex = perm_explainer(path, m, Xbg_np)
    v, b, fx = perm_rows(ex, f, Xc_np, ids, seed)
    return k, v, b, fx, time.process_time() - c0, time.perf_counter() - t0


def predict_p(pipe, X):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=DeprecationWarning)
        return pipe.predict_proba(X)[:, 1]


def shap_frame(rows, V, Bv, FX):
    df = pd.DataFrame(V, columns=FEATURES)
    df.insert(0, "additivity_error", np.abs(Bv + V.sum(axis=1) - FX))
    df.insert(0, "model_output", FX)
    df.insert(0, "base_value", Bv)
    df.insert(0, "row_id", rows)
    return df


def csv_bytes(df):
    return df.to_csv(index=False, float_format="%.17g", lineterminator="\n").encode("utf-8")

In [5]:
# Cell 5 - index sets per split (split 1: Stage 1 files; splits 2-5: the R4 index files), as R5
IDX = {}
for k in SPLITS:
    if k == 1:
        dev = pd.read_csv(os.path.join(IDX1, "dev_folds.csv")).row_id.to_numpy()
        bg = pd.read_csv(os.path.join(IDX1, "shap_background_500_dev.csv")).row_id.to_numpy()
        bg100 = pd.read_csv(os.path.join(IDX1, "shap_perm_background_100_dev.csv")).row_id.to_numpy()
        core = pd.read_csv(os.path.join(IDX1, "shap_perm_core_500_test.csv")).row_id.to_numpy()
        test = PRED.row_id.to_numpy()
    else:
        p = os.path.join(R4, "indices", f"split{k}")
        dev = pd.read_csv(os.path.join(p, "dev_folds.csv")).row_id.to_numpy()
        test = pd.read_csv(os.path.join(p, "test_rows.csv")).row_id.to_numpy()
        bg = pd.read_csv(os.path.join(p, "shap_background_500_dev.csv")).row_id.to_numpy()
        bg100 = pd.read_csv(os.path.join(p, "shap_perm_background_100_dev.csv")).row_id.to_numpy()
        core = pd.read_csv(os.path.join(p, "test_core_500.csv")).row_id.to_numpy()
    assert len(core) == 500 and int(y_all[core].sum()) == 70 and len(bg) == 500 and len(bg100) == 100
    assert set(bg100) <= set(bg) <= set(dev) and set(core) <= set(test) and not set(dev) & set(test)
    IDX[k] = {"dev": np.sort(dev), "test": np.sort(test), "bg": np.sort(bg), "bg100": np.sort(bg100), "core": np.sort(core)}
    print(f"split {k}: dev {len(dev)}, test {len(test)}, background 500/100, core 500 (70 positive)")
XN = X_all.to_numpy()

split 1: dev 202944, test 50736, background 500/100, core 500 (70 positive)
split 2: dev 202944, test 50736, background 500/100, core 500 (70 positive)
split 3: dev 202944, test 50736, background 500/100, core 500 (70 positive)
split 4: dev 202944, test 50736, background 500/100, core 500 (70 positive)
split 5: dev 202944, test 50736, background 500/100, core 500 (70 positive)


In [6]:
# Cell 6 - checks against stored R5 outputs, BEFORE any refit (design section 12 item 11)
REG_PATH = os.path.join(OUT, "r5_regression_check.csv")


def r5_shap(k, s, m):
    rec = read_json(os.path.join(R5, f"split{k}", s, f"SHAP_{m}.done.json"))
    return pd.read_csv(os.path.join(R5, rec["file"]), float_precision="round_trip").set_index("row_id")


def worker_rows(path, m, Xbg_np, Xc_np, ids, seed, n_workers):
    # the worker path of Cell 9 on a few rows: one row per chunk, so every row goes through a worker process
    jobs = (delayed(perm_chunk)(path, m, Xbg_np, Xc_np[i:i + 1], ids[i:i + 1], seed, i) for i in range(len(ids)))
    got = {c: (v, b) for c, v, b, fx, _, _ in Parallel(n_jobs=n_workers)(jobs)}
    return np.vstack([got[i][0] for i in range(len(ids))]), np.concatenate([got[i][1] for i in range(len(ids))])


if os.path.exists(REG_PATH):
    REG = pd.read_csv(REG_PATH)
    print("R5 checks passed earlier:\n" + REG.to_string(index=False))
else:
    reg = []
    for k in SPLITS:
        I = IDX[k]
        core, ids = I["core"], I["core"][:REG_ROWS]
        Xbg100, Xbg500 = X_all.iloc[I["bg100"]].reset_index(drop=True), X_all.iloc[I["bg"]].reset_index(drop=True)
        Xc = X_all.iloc[ids].reset_index(drop=True)
        for m in (MEMBERS + SPLIT1_EXTRA if k == 1 else MEMBERS):
            path = R5_MODEL[(k, m)]
            C, N = r5_shap(k, "common", m), r5_shap(k, "native", m)
            chk = np.array([r for r in core if r in C.index and r in N.index])   # all 500 on the PC (test: fewer rows)
            if not TEST_MODE and len(chk) != 500:
                raise SystemExit(f"split {k} {m}: the R5 SHAP files do not cover the 500-row core")
            pipe = load_pipe(path, m)
            # (a) the model reproduces the stored core outputs (probability: common file; native scale: native file)
            d_c = float(np.abs(predict_p(pipe, X_all.iloc[chk]) - C.loc[chk, "model_output"].to_numpy()).max())
            d_n = float(np.abs(np.asarray(model_output(m, pipe, X_all.iloc[chk]), float) - N.loc[chk, "model_output"].to_numpy()).max())
            # (b) R6 code on the first REG_ROWS core rows = stored R5 SHAP values
            f, ex = perm_explainer(path, m, Xbg100.to_numpy())
            v, b, _ = perm_rows(ex, f, Xc.to_numpy(), ids, perm_seed(k))
            dv_c = float(np.abs(v - C.loc[ids, FEATURES].to_numpy()).max()); db_c = float(np.abs(b - C.loc[ids, "base_value"].to_numpy()).max())
            if m == "MLP":
                vn, bn = v, b
                tol_n = 0.0
            elif m == "LR":
                vn, bn, _, _ = linear_all(path, Xc, Xbg500); tol_n = 1e-10
            else:
                _, vn, bn, _, _, _ = tree_chunk(path, m, Xc, Xbg500, 0); tol_n = 1e-10
            dv_n = float(np.abs(vn - N.loc[ids, FEATURES].to_numpy()).max()); db_n = float(np.abs(bn - N.loc[ids, "base_value"].to_numpy()).max())
            mlp_same = bool(m != "MLP" or (C.loc[chk, FEATURES + ["base_value"]].equals(N.loc[chk, FEATURES + ["base_value"]])))
            # (c) worker path = single-process path, byte for byte (permutation models run in workers in Cell 9)
            wk = "not used"
            if m in PERM_WORKERS:
                vw, bw = worker_rows(path, m, Xbg100.to_numpy(), Xc.to_numpy(), ids, perm_seed(k), min(PERM_WORKERS[m], len(ids)))
                wk = bool(vw.tobytes() == v.tobytes() and bw.tobytes() == b.tobytes())
            ok = (d_c <= 1e-12 and d_n <= 1e-12 and dv_c == 0 and db_c == 0 and dv_n <= tol_n and db_n <= tol_n and mlp_same)
            reg.append({"split": k, "model": m, "rows": len(ids), "core_rows_checked": len(chk), "core_output_prob_max_diff": d_c, "core_output_native_max_diff": d_n,
                        "common_values_max_diff": dv_c, "common_base_max_diff": db_c, "native_values_max_diff": dv_n,
                        "native_base_max_diff": db_n, "native_tolerance": tol_n, "r5_mlp_native_equals_common": mlp_same,
                        "worker_path_byte_identical": wk, "pass": bool(ok)})
            print(reg[-1], flush=True)
    REG = pd.DataFrame(reg)
    if not REG["pass"].all():
        raise SystemExit("R5 checks failed: R6 does not reproduce the stored R5 outputs " + str(REG[~REG["pass"]][["split", "model"]].values.tolist()))
    write_atomic(REG_PATH, REG.to_csv(index=False, float_format="%.3g", lineterminator="\n").encode("utf-8"))
# a model whose worker path was not byte-identical runs single-process
for m in list(PERM_WORKERS):
    w = REG[REG.model == m].worker_path_byte_identical.astype(str)
    if len(w) and not (w == "True").all():
        print(f"{m}: worker path not byte-identical on this machine; its permutation rows run single-process.")
        PERM_WORKERS.pop(m)
print("permutation workers used:", PERM_WORKERS)

{'split': 1, 'model': 'CAT', 'rows': 5, 'core_rows_checked': 500, 'core_output_prob_max_diff': 0.0, 'core_output_native_max_diff': 0.0, 'common_values_max_diff': 0.0, 'common_base_max_diff': 0.0, 'native_values_max_diff': 0.0, 'native_base_max_diff': 0.0, 'native_tolerance': 1e-10, 'r5_mlp_native_equals_common': True, 'worker_path_byte_identical': True, 'pass': True}
{'split': 1, 'model': 'XGB', 'rows': 5, 'core_rows_checked': 500, 'core_output_prob_max_diff': 0.0, 'core_output_native_max_diff': 0.0, 'common_values_max_diff': 0.0, 'common_base_max_diff': 0.0, 'native_values_max_diff': 0.0, 'native_base_max_diff': 0.0, 'native_tolerance': 1e-10, 'r5_mlp_native_equals_common': True, 'worker_path_byte_identical': 'not used', 'pass': True}
{'split': 1, 'model': 'LGBM', 'rows': 5, 'core_rows_checked': 500, 'core_output_prob_max_diff': 0.0, 'core_output_native_max_diff': 0.0, 'common_values_max_diff': 0.0, 'common_base_max_diff': 0.0, 'native_values_max_diff': 0.0, 'native_base_max_diff': 0.

In [7]:
# Cell 7 - seeds, refit plan, Rashomon set and RF subset, written to seeds_r6.json BEFORE any fit
def config_of(k, m):
    return int(FZ["models"][m]["config_id"]) if k == 1 else int(read_json(os.path.join(R4, f"split{k}", m, "summary.json"))["config_id"])


def boot_rows(k, seed):
    # within-class resampling of the development rows with replacement (class sizes kept), sorted
    dev = IDX[k]["dev"]; rng = np.random.default_rng(seed)
    parts = []
    for c in (0, 1):
        pool = dev[y_all[dev] == c]
        parts.append(pool[rng.integers(0, len(pool), len(pool))])
    return np.sort(np.concatenate(parts), kind="stable")


UNITS = []
def unit(k, kind, i, m, cid, est_seed, rs_seed, settings):
    name = f"split{k}_{kind}{i:02d}_{m}" if kind != "rash" else f"split{k}_rash_c{cid:02d}_{m}"
    UNITS.append({"unit": name, "split": k, "kind": kind, "index": i, "model": m, "config_id": cid,
                  "estimator_seed": est_seed, "resample_seed": rs_seed, "settings": settings})


W = {k: r6_words(k) for k in SPLITS}
# (a) member bootstrap refits, split 1 then splits 2-5; (b) split-1 LR and RF bootstrap refits
for k in SPLITS:
    for i in range(1, N_BOOT[k] + 1):
        for m in MEMBERS:
            unit(k, "boot", i, m, config_of(k, m), tc.MODEL_SEED, W[k][i - 1], ["common", "native"])
if 1 in SPLITS:
    for i in range(1, N_BOOT[1] + 1):
        for m in SPLIT1_EXTRA:
            unit(1, "boot", i, m, config_of(1, m), tc.MODEL_SEED, W[1][i - 1], ["common", "native"])
    # (c) seed-only refits: full development data, new estimator seed
    for i in range(1, N_SEED + 1):
        for m in SEED_MODELS:
            unit(1, "seed", i, m, config_of(1, m), W[1][10 + i - 1] % (2 ** 31 - 1), None, ["native"])
    # (d) Rashomon set: configurations within RASH_MARGIN of the best mean CV AP, best first (ties: lower config id),
    # at most RASH_MAX per model including the selected configuration (explained in R5, not refitted)
    RASH = {}
    for m in RASH_MODELS:
        cv = CV1[m].sort_values(["mean_test_ap", "config_id"], ascending=[False, True])
        sel = cv[cv.mean_test_ap >= cv.mean_test_ap.max() - RASH_MARGIN].head(RASH_MAX)
        assert int(sel.config_id.iloc[0]) == config_of(1, m)
        RASH[m] = [{"config_id": int(r.config_id), "mean_cv_ap": float(r.mean_test_ap), "gap_to_best": float(cv.mean_test_ap.max() - r.mean_test_ap)}
                   for r in sel.itertuples()]
        for r in RASH[m][1:]:
            unit(1, "rash", 0, m, r["config_id"], tc.MODEL_SEED, None, ["native"])
    sub = train_test_split(IDX[1]["core"], train_size=RF_SUBSET_N, stratify=y_all[IDX[1]["core"]], random_state=W[1][15])[0]
    RF_SUBSET = np.sort(sub)
    assert int(y_all[RF_SUBSET].sum()) == RF_SUBSET_POS
for u in UNITS:
    if u["kind"] == "boot":
        r = boot_rows(u["split"], u["resample_seed"])
        u.update({"rows": int(len(r)), "unique_rows": int(len(np.unique(r))), "positives": int(y_all[r].sum()),
                  "rows_sha256": sha_bytes(r.astype(np.int64).tobytes())})
    else:
        u.update({"rows": int(len(IDX[u["split"]]["dev"])), "unique_rows": int(len(IDX[u["split"]]["dev"])),
                  "positives": int(y_all[IDX[u["split"]]["dev"]].sum()), "rows_sha256": "full development data"})
SEEDS = {"stream": "paper_r6 = SeedSequence(42).generate_state(16)[15] (16th stream, append-only; D39)", "paper_r6": PAPER_R6,
         "per_split_words": {str(k): W[k] for k in SPLITS},
         "word_use": {"0-9": "bootstrap resampling seeds of refits 1-10 (np.random.default_rng(word); within-class resampling of the development rows)",
                      "10-14": "split-1 seed-only estimator seeds 1-5 (word mod 2**31-1)", "15": "split-1 RF fallback 200-row subset (train_test_split, stratified)",
                      "16": "H2 core-row bootstrap (post-hoc, r5_h2_boot.py)"},
         "bootstrap_estimator_seed": tc.MODEL_SEED, "permutation_seeds": {str(k): perm_seed(k) for k in SPLITS},
         "units": UNITS, "test_mode": TEST_MODE}
if 1 in SPLITS:
    SEEDS.update({"rashomon": {"margin": RASH_MARGIN, "max_per_model_incl_selected": RASH_MAX, "source": "Stage 3 cv_results (mean CV AP; MLP on the 50k subsample, D14)",
                               "sets": RASH, "cv_results_sha256": {m: sha(os.path.join(PROJECT_DIR, "stage3_outputs", "models", m, f"cv_results_{m}.csv")) for m in RASH_MODELS}},
                  "rf_subset_200": {"seed": W[1][15], "rows": [int(r) for r in RF_SUBSET], "positives": RF_SUBSET_POS,
                                    "rule": "used only if the first RF refit's native TreeSHAP exceeds 8 min with 6 workers (section 9 item 11)"}})
SEEDS_PATH = os.path.join(OUT, "seeds_r6.json")
blob = json.dumps(SEEDS, indent=1).encode("utf-8")
if os.path.exists(SEEDS_PATH):
    if open(SEEDS_PATH, "rb").read() != blob:
        raise SystemExit("seeds_r6.json differs from the seeds derived now: settings changed between sessions. Do not mix runs.")
    print("seeds_r6.json unchanged (resume)")
else:
    write_atomic(SEEDS_PATH, blob)
print(f"seeds_r6.json SHA-256 {sha(SEEDS_PATH)}; {len(UNITS)} refits planned")
print(pd.DataFrame(UNITS)[["unit", "config_id", "estimator_seed", "resample_seed", "unique_rows", "positives"]].to_string(index=False))
if 1 in SPLITS:
    print("Rashomon set:", {m: [r["config_id"] for r in v] for m, v in RASH.items()})

seeds_r6.json SHA-256 1b5e3b00bb3aaeabdc94614b7f9d5ff94ba55fdae6b7ff79498e6741a847f82d; 219 refits planned
                unit  config_id  estimator_seed  resample_seed  unique_rows  positives
   split1_boot01_CAT          7        22980462   4158215418.0       128488      28277
   split1_boot01_XGB          2        22980462   4158215418.0       128488      28277
  split1_boot01_LGBM         19        22980462   4158215418.0       128488      28277
   split1_boot01_MLP         21        22980462   4158215418.0       128488      28277
   split1_boot02_CAT          7        22980462    959507536.0       128232      28277
   split1_boot02_XGB          2        22980462    959507536.0       128232      28277
  split1_boot02_LGBM         19        22980462    959507536.0       128232      28277
   split1_boot02_MLP         21        22980462    959507536.0       128232      28277
   split1_boot03_CAT          7        22980462   2368656717.0       128161      28277
   split1_boot03_XGB   

In [8]:
# Cell 8 - task list and plan (planning rates from timing_r5.csv on the PC; seconds; not results)
# medians over the R5 PC tasks (r6_time_estimate.py)
FIT_S = {"CAT": 21, "XGB": 4, "LGBM": 4, "MLP": 15, "RF": 15, "LR": 7}                  # R5 Cell 6 (fit + full-test prediction)
PERM_S = {"CAT": 0.110, "XGB": 0.119, "LGBM": 0.096, "MLP": 0.120, "RF": 0.243, "LR": 0.108}   # per row, one process
TREE_S = {"CAT": 0.121, "XGB": 0.022, "LGBM": 0.080, "RF": 0.683}                       # per row, 6 workers


def rf_fallback():
    p = os.path.join(OUT, "rf_fallback.json")
    return read_json(p)["fallback"] if os.path.exists(p) else None


def task_rows(u):
    rows = IDX[u["split"]]["core"]
    if u["model"] == "RF" and rf_fallback() and not u.get("first_rf"):
        rows = RF_SUBSET
    return rows[:MAX_ROWS] if MAX_ROWS else rows


FIRST_RF = next((u["unit"] for u in UNITS if u["model"] == "RF"), None)
for u in UNITS:
    u["first_rf"] = u["unit"] == FIRST_RF


def explainer_kind(u, s):
    if s == "common" or u["model"] == "MLP":
        return "perm"
    return "linear" if u["model"] == "LR" else "tree"


TASKS = [(u, s) for u in UNITS for s in (["native", "common"] if len(u["settings"]) == 2 else u["settings"])]
# native first, so the RF fallback is decided before the first RF common task; MLP native is a copy of MLP common
TASKS = [(u, s) for u, s in TASKS if not (u["model"] == "MLP" and s == "native" and "common" in u["settings"])]


def task_name(u, s):
    return f"{u['unit']}_{s}"


def task_files(u, s):
    rel = os.path.join(f"split{u['split']}", s, f"SHAP_{u['unit'][len(f'split{u['split']}_'):]}.csv")
    return rel, os.path.join(OUT, rel[:-4] + ".done.json")


def is_done(u, s):
    rel, dp = task_files(u, s)
    if not os.path.exists(dp):
        return False
    if sha(os.path.join(OUT, rel)) != read_json(dp)["sha256"]:
        raise SystemExit(f"{rel} differs from its .done.json; delete both and rerun this task.")
    return True


def planned_task_s(u, s):
    m, n = u["model"], len(task_rows(u))
    kind = explainer_kind(u, s)
    if kind == "linear":
        return 2
    if kind == "tree":
        return n * TREE_S[m] + 5
    w = PERM_WORKERS.get(m, 1)
    return n * PERM_S[m] / (0.8 * w if w > 1 else 1) + 5


def unit_done(u):
    return os.path.exists(os.path.join(MODELS_DIR, u["unit"] + ".done.json"))


PLAN = pd.DataFrame([{"unit": u["unit"], "kind": u["kind"], "model": u["model"], "fit": "done" if unit_done(u) else "to fit",
                      "tasks": "+".join(s for uu, s in TASKS if uu is u), "planned_min": (0 if unit_done(u) else FIT_S[u["model"]]) / 60
                      + sum(planned_task_s(uu, s) for uu, s in TASKS if uu is u and not is_done(uu, s)) / 60} for u in UNITS])
print(PLAN.groupby(["kind", "model"]).agg(units=("unit", "size"), planned_min=("planned_min", "sum")).round(1).to_string())
todo = PLAN.planned_min.sum()
part_a = PLAN[(PLAN.kind == "boot") & PLAN.model.isin(MEMBERS)].planned_min.sum()
print(f"\nPlanned time for the remaining work: {todo:.0f} min (about {todo / 60:.1f} h; decisive part (a) {part_a / 60:.1f} h); allow up to about 1.5x.")

            units  planned_min
kind model                    
boot CAT       30         54.3
     LGBM      30         51.0
     LR        10          4.2
     MLP       30         40.0
     RF        10         65.3
     XGB       30         42.2
rash CAT        9         13.0
     LGBM       9          7.4
     LR         9          1.3
     MLP        9         12.0
     RF         9         54.2
     XGB        9          3.0
seed CAT        5          7.2
     LGBM       5          4.1
     MLP        5          6.7
     RF         5         30.1
     XGB        5          1.7

Planned time for the remaining work: 398 min (about 6.6 h; decisive part (a) 3.1 h); allow up to about 1.5x.


In [9]:
# Cell 9 - refits and SHAP tasks (resumable). Progress and the time remaining are printed as chunks finish.
SPEED, LAST = [], [0.0]


def progress(msg, final=False):
    now = time.perf_counter()
    if final or now - LAST[0] >= 30:
        print(msg, flush=True); LAST[0] = now


def stop_hook(tag):                               # test hook: simulated interruption
    if STOP_AFTER and STOP_AFTER == tag:
        raise SystemExit(f"TEST INTERRUPT after {tag}")


def fit_unit(u):
    mp, dp = os.path.join(MODELS_DIR, u["unit"] + ".joblib"), os.path.join(MODELS_DIR, u["unit"] + ".done.json")
    if os.path.exists(dp):
        rec = read_json(dp)
        if sha(mp) != rec["model_sha256"]:
            raise SystemExit(f"{mp} changed since it was fitted; delete it and {dp} to refit.")
        return mp
    k, m = u["split"], u["model"]
    rows = boot_rows(k, u["resample_seed"]) if u["kind"] == "boot" else IDX[k]["dev"]
    assert (sha_bytes(rows.astype(np.int64).tobytes()) == u["rows_sha256"]) if u["kind"] == "boot" else True
    pipe = tc.make_pipeline(m, n_jobs=CAT_THREADS if m == "CAT" else -1)
    pipe.set_params(**tc.sampled_configs(m)[u["config_id"]])
    if u["kind"] == "seed":
        pipe.set_params(**({"clf__random_seed": u["estimator_seed"]} if m == "CAT" else {"clf__random_state": u["estimator_seed"]}))
    t0 = time.perf_counter()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        pipe.fit(X_all.iloc[rows], y_all[rows])
    fit_s = time.perf_counter() - t0
    tmp = mp + ".tmp"
    joblib.dump(pipe, tmp); os.replace(tmp, mp)
    pt = predict_p(load_pipe(mp, m), X_all.iloc[IDX[k]["test"]])          # descriptive only (refit spread); nothing is selected on it
    yt = y_all[IDX[k]["test"]]
    rec = {"unit": u["unit"], "model_sha256": sha(mp), "fit_s": fit_s, "rows_sha256": u["rows_sha256"], "config_id": u["config_id"],
           "estimator_seed": u["estimator_seed"], "test_ap": float(average_precision_score(yt, pt)), "test_roc_auc": float(roc_auc_score(yt, pt)),
           "machine": MACHINE, "finished_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}
    write_atomic(dp, json.dumps(rec, indent=1).encode("utf-8"))
    print(f"[{u['unit']}] fitted in {fit_s:.0f} s; test AP {rec['test_ap']:.4f}", flush=True)
    stop_hook(f"fit:{u['unit']}")
    return mp


def manifest(u, s, rows, chunk, bg_key, bg, seed, mpath):
    wd = os.path.join(WORK, task_name(u, s))
    os.makedirs(wd, exist_ok=True)
    man = {"rows_sha256": sha_bytes(np.asarray(rows, np.int64).tobytes()), "n_rows": len(rows), "chunk": chunk,
           "model_sha256": sha(mpath), "background": bg_key, "background_sha256": sha_bytes(np.asarray(bg, np.int64).tobytes()),
           "seed": seed, "max_evals": MAX_EVALS, "shap": shap.__version__}
    mp = os.path.join(wd, "manifest.json")
    if os.path.exists(mp):
        if read_json(mp) != man:
            raise SystemExit(f"{mp} was written with other settings. Delete {wd} and rerun.")
    else:
        write_atomic(mp, json.dumps(man, indent=1).encode("utf-8"))
    return wd


def load_chunks(wd, n):
    got = {}
    for c in range(n):
        p = os.path.join(wd, f"c{c:04d}.npz")
        if os.path.exists(p):
            with np.load(p) as z:
                got[c] = {key: z[key] for key in z.files}
    return got


def save_chunk(wd, c, **arrs):
    p = os.path.join(wd, f"c{c:04d}.npz")
    tmp = p + ".tmp.npz"
    np.savez(tmp, **arrs)
    os.replace(tmp, p)


def finish(u, s, rows, V, Bv, FX, kind, bg, seed, mpath, workers, wall, cpu, chunks):
    df = shap_frame(rows, V, Bv, FX)
    blob = csv_bytes(df)
    rel, dp = task_files(u, s)
    write_atomic(os.path.join(OUT, rel), blob)
    scale = "probability" if s == "common" else SCALE_NATIVE[u["model"]]
    expl = {"linear": "LinearExplainer (interventional, 500 bg, standardized inputs)", "tree": f"TreeExplainer (interventional, {len(bg)} bg)",
            "perm": f"PermutationExplainer ({len(bg)} bg, max_evals {MAX_EVALS})"}[kind]
    rec = {"file": rel.replace(os.sep, "/"), "sha256": sha_bytes(blob), "unit": u["unit"], "split": u["split"], "kind": u["kind"],
           "model": u["model"], "config_id": u["config_id"], "setting": s, "rows": len(rows), "row_set": "rf_subset_200" if len(rows) == RF_SUBSET_N else "core",
           "explainer": expl, "output_scale": scale, "seed": seed, "model_sha256": sha(mpath), "chunks": chunks, "workers": workers,
           "wall_s": wall, "cpu_s": cpu, "max_additivity_error": float(df.additivity_error.max()),
           "mean_additivity_error": float(df.additivity_error.mean()), "machine": MACHINE,
           "finished_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}
    write_atomic(dp, json.dumps(rec, indent=1).encode("utf-8"))
    return rec, blob


def run_task(u, s, mpath):
    k, m = u["split"], u["model"]
    kind = explainer_kind(u, s)
    bg_key = "bg100" if kind == "perm" else "bg"
    bg = IDX[k][bg_key]
    rows = task_rows(u)
    seed = perm_seed(k) if kind == "perm" else None
    chunk = 10 ** 9 if kind == "linear" else TREE_CHUNK if kind == "tree" else PERM_CHUNK
    wd = manifest(u, s, rows, chunk, bg_key, bg, seed, mpath)
    Xr, Xbg = X_all.iloc[rows].reset_index(drop=True), X_all.iloc[bg].reset_index(drop=True)
    starts = list(range(0, len(rows), chunk))
    got = load_chunks(wd, len(starts))
    todo = [c for c in range(len(starts)) if c not in got]
    t0, n_left, done_here = time.perf_counter(), sum(len(rows[starts[c]:starts[c] + chunk]) for c in todo), 0
    workers = TREE_WORKERS if kind == "tree" else PERM_WORKERS.get(m, 1) if kind == "perm" else 1

    def saved(c, v, b, fx, cpu, w):
        nonlocal n_left, done_here
        save_chunk(wd, c, v=v, b=b, fx=fx, wall=np.array([w]), cpu=np.array([cpu]))
        n_left -= len(v); done_here += len(v)
        el = time.perf_counter() - t0
        progress(f"  [{task_name(u, s)}] {len(rows) - n_left}/{len(rows)} rows, {el / 60:.1f} min; task ETA {n_left * el / done_here / 60:.1f} min", n_left == 0)
        stop_hook(f"{task_name(u, s)}:{c}")

    if kind == "linear":
        if todo:
            v, b, fx, w = linear_all(mpath, Xr, Xbg)
            saved(0, v, b, fx, w, w)
    elif kind == "tree":
        jobs = (delayed(tree_chunk)(mpath, m, Xr.iloc[starts[c]:starts[c] + chunk], Xbg, c) for c in todo)
        for c, v, b, fx, cpu, w in Parallel(n_jobs=TREE_WORKERS, return_as="generator_unordered")(jobs):
            saved(c, v, b, fx, cpu, w)
    elif workers > 1:
        Xn, Xb = Xr.to_numpy(), Xbg.to_numpy()
        jobs = (delayed(perm_chunk)(mpath, m, Xb, Xn[starts[c]:starts[c] + chunk], rows[starts[c]:starts[c] + chunk], seed, c) for c in todo)
        for c, v, b, fx, cpu, w in Parallel(n_jobs=workers, return_as="generator_unordered")(jobs):
            saved(c, v, b, fx, cpu, w)
    else:
        f, ex = perm_explainer(mpath, m, Xbg.to_numpy())
        Xn = Xr.to_numpy()
        for c in todo:
            sl = slice(starts[c], starts[c] + chunk)
            c0, w0 = time.process_time(), time.perf_counter()
            v, b, fx = perm_rows(ex, f, Xn[sl], rows[sl], seed)
            saved(c, v, b, fx, time.process_time() - c0, time.perf_counter() - w0)
    got = load_chunks(wd, len(starts))
    V = np.vstack([got[c]["v"] for c in range(len(starts))])
    Bv = np.concatenate([got[c]["b"] for c in range(len(starts))])
    FX = np.concatenate([got[c]["fx"] for c in range(len(starts))])
    cpu = float(sum(got[c]["cpu"][0] for c in got))
    if workers == 1:
        wall = float(sum(got[c]["wall"][0] for c in got))
    elif todo == list(range(len(starts))):
        wall = time.perf_counter() - t0
    else:
        wall = float(sum(got[c]["wall"][0] for c in got)) / workers
    rec, blob = finish(u, s, rows, V, Bv, FX, kind, bg, seed, mpath, workers, wall, cpu, len(starts))
    if m == "MLP" and s == "common" and "native" in u["settings"]:
        # MLP native = common (same explainer, background, scale and seed): the same values, written once more
        rec_n, blob_n = finish(u, "native", rows, V, Bv, FX, kind, bg, seed, mpath, workers, 0.0, 0.0, len(starts))
        assert blob_n == blob and rec_n["sha256"] == rec["sha256"]
    if m == "RF" and s == "native" and u["first_rf"] and rf_fallback() is None:
        fb = {"first_rf_task": task_name(u, s), "rows": len(rows), "workers": TREE_WORKERS, "wall_s": wall, "threshold_s": RF_FALLBACK_S,
              "fallback": bool(wall > RF_FALLBACK_S), "decided_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}
        if workers != 6 and not TEST_MODE:
            raise SystemExit("the RF fallback rule is stated for 6 workers")
        write_atomic(os.path.join(OUT, "rf_fallback.json"), json.dumps(fb, indent=1).encode("utf-8"))
        print(f"RF fallback decided: first RF native TreeSHAP {wall / 60:.1f} min vs {RF_FALLBACK_S / 60:.0f} min -> "
              f"{'200-row subset for every later RF task' if fb['fallback'] else 'all 500 core rows'}", flush=True)
    return rec


t_session = time.perf_counter()
for i, u in enumerate(UNITS):
    ut = [s for uu, s in TASKS if uu is u]
    if unit_done(u) and all(is_done(u, s) for s in ut):
        continue
    t1 = time.perf_counter()
    mpath = fit_unit(u)
    for s in ut:
        if is_done(u, s):
            continue
        t2 = time.perf_counter()
        rec = run_task(u, s, mpath)
        SPEED.append((time.perf_counter() - t2) / planned_task_s(u, s))
        print(f"[{task_name(u, s)}] {rec['rows']} rows in {(time.perf_counter() - t2) / 60:.2f} min; max additivity error {rec['max_additivity_error']:.2e}", flush=True)
    f = float(np.median(SPEED)) if SPEED else 1.0
    rest = sum((0 if unit_done(v) else FIT_S[v["model"]]) + sum(planned_task_s(v, s) for vv, s in TASKS if vv is v and not is_done(v, s)) for v in UNITS[i + 1:])
    print(f"== {u['unit']} done in {(time.perf_counter() - t1) / 60:.1f} min; remaining about {rest * f / 60:.0f} min "
          f"({len(UNITS) - i - 1} refits); session {(time.perf_counter() - t_session) / 60:.0f} min", flush=True)
print(f"All {len(UNITS)} refits and {len(TASKS)} SHAP tasks finished. This session: {(time.perf_counter() - t_session) / 60:.1f} min.")

[split1_boot01_CAT] fitted in 20 s; test AP 0.4285
  [split1_boot01_CAT_native] 50/500 rows, 0.3 min; task ETA 2.8 min
  [split1_boot01_CAT_native] 500/500 rows, 0.6 min; task ETA 0.0 min
[split1_boot01_CAT_native] 500 rows in 0.56 min; max additivity error 7.53e-08
  [split1_boot01_CAT_common] 500/500 rows, 0.4 min; task ETA 0.0 min
[split1_boot01_CAT_common] 500 rows in 0.43 min; max additivity error 1.89e-15
== split1_boot01_CAT done in 1.3 min; remaining about 330 min (218 refits); session 1 min
[split1_boot01_XGB] fitted in 7 s; test AP 0.4272
  [split1_boot01_XGB_native] 50/500 rows, 0.4 min; task ETA 3.9 min
  [split1_boot01_XGB_native] 500/500 rows, 0.7 min; task ETA 0.0 min
[split1_boot01_XGB_native] 500 rows in 0.70 min; max additivity error 6.42e-06
  [split1_boot01_XGB_common] 300/500 rows, 0.5 min; task ETA 0.3 min
  [split1_boot01_XGB_common] 500/500 rows, 0.8 min; task ETA 0.0 min
[split1_boot01_XGB_common] 500 rows in 0.84 min; max additivity error 2.22e-16
== split1_bo

In [10]:
# Cell 10 - refit table, additivity, timing, environment log, report
ref, add, tim = [], [], []
for u in UNITS:
    rec = read_json(os.path.join(MODELS_DIR, u["unit"] + ".done.json"))
    ref.append({**{k_: u[k_] for k_ in ["unit", "split", "kind", "index", "model", "config_id", "estimator_seed", "resample_seed", "unique_rows", "positives"]},
                "model_sha256": rec["model_sha256"], "fit_s": rec["fit_s"], "test_ap": rec["test_ap"], "test_roc_auc": rec["test_roc_auc"]})
    for s in u["settings"]:
        rel, dp = task_files(u, s)
        r = read_json(dp)
        add.append({"unit": u["unit"], "split": u["split"], "kind": u["kind"], "model": u["model"], "setting": s, "rows": r["rows"],
                    "row_set": r["row_set"], "explainer": r["explainer"], "output_scale": r["output_scale"], "max_abs_additivity_error": r["max_additivity_error"]})
        tim.append({"task": task_name(u, s), "rows": r["rows"], "workers": r["workers"], "wall_s": r["wall_s"], "cpu_s": r["cpu_s"], "machine": r["machine"]})
REF, ADD, TIM = pd.DataFrame(ref), pd.DataFrame(add), pd.DataFrame(tim)
write_atomic(os.path.join(OUT, "refits_r6.csv"), REF.to_csv(index=False, float_format="%.10g", lineterminator="\n").encode("utf-8"))
write_atomic(os.path.join(OUT, "additivity_r6.csv"), ADD.to_csv(index=False, float_format="%.6g", lineterminator="\n").encode("utf-8"))
write_atomic(os.path.join(OUT, "timing_r6.csv"), TIM.to_csv(index=False, float_format="%.6g", lineterminator="\n").encode("utf-8"))
write_atomic(os.path.join(OUT, "gates_r6.csv"), GATE_DF.to_csv(index=False, lineterminator="\n").encode("utf-8"))
expected = sum(len(u["settings"]) for u in UNITS)
if len(ADD) != expected or len(REF) != len(UNITS):
    raise SystemExit(f"{len(ADD)} SHAP files and {len(REF)} refits, expected {expected} and {len(UNITS)}")
from importlib import metadata
env = {"machine": MACHINE, "python": platform.python_version(), "platform": platform.platform(), "cpu_count": os.cpu_count(),
       "packages": {p: metadata.version(p) for p in ["numpy", "pandas", "scipy", "scikit-learn", "xgboost", "lightgbm", "catboost", "shap", "joblib"]},
       "pins_differ": {k_: list(v) for k_, v in tc.check_pins(PROJECT_DIR).items()}, "test_mode": TEST_MODE, "splits": SPLITS,
       "n_boot": N_BOOT, "n_seed": N_SEED, "rashomon_max": RASH_MAX, "tree_workers": TREE_WORKERS, "perm_workers": PERM_WORKERS,
       "max_rows": MAX_ROWS, "paper_r6": PAPER_R6, "seeds_r6_sha256": sha(SEEDS_PATH),
       "rf_fallback": read_json(os.path.join(OUT, "rf_fallback.json")) if os.path.exists(os.path.join(OUT, "rf_fallback.json")) else None,
       "written_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}
write_atomic(os.path.join(OUT, f"env_log_r6_{MACHINE}.json"), json.dumps(env, indent=1).encode("utf-8"))
pd.set_option("display.width", 200)
print(REF.groupby(["split", "kind", "model"]).agg(refits=("unit", "size"), test_ap_mean=("test_ap", "mean"), test_ap_sd=("test_ap", "std"),
                                                    fit_s=("fit_s", "mean")).round(4).to_string())
print(f"\nmax additivity error: tree/linear {ADD[~ADD.explainer.str.startswith('Perm')].max_abs_additivity_error.max():.2e}; "
      f"permutation {ADD[ADD.explainer.str.startswith('Perm')].max_abs_additivity_error.max():.2e}")
print("RF fallback:", env["rf_fallback"])
print("\nNext: run  python pack_r6_outputs.py  and upload r6_upload/ (every zip and SHA256SUMS_r6.txt).")

                  refits  test_ap_mean  test_ap_sd    fit_s
split kind model                                           
1     boot CAT        10        0.4288      0.0010  21.1456
           LGBM       10        0.4281      0.0016   3.5893
           LR         10        0.3998      0.0003   1.3468
           MLP        10        0.4085      0.0115  15.7370
           RF         10        0.4224      0.0019  11.7450
           XGB        10        0.4267      0.0019   6.7336
      rash CAT         9        0.4311      0.0005  13.7805
           LGBM        9        0.4310      0.0009   3.2069
           LR          9        0.3999      0.0005   6.2784
           MLP         9        0.4279      0.0024  15.3015
           RF          9        0.4271      0.0014  13.8700
           XGB         9        0.4308      0.0008   4.6839
      seed CAT         5        0.4316      0.0003  21.0333
           LGBM        5        0.4314      0.0002   3.7868
           MLP         5        0.4246  